# XGBoost active learning — traditional, function-based notebook

Plain procedural ML: **functions, no user-defined classes**. The flow:

1. **Config** — all paths, tunable `params`, and the `label_dict` in one cell.
2. **Preprocess** (pandas + `tqdm`) — embed the text into `emb_*` feature
   columns, keep everything in a **DataFrame**, and **save**
   `train_features_dataset_processed` + the processed test set to parquet.
3. **Load & run** (next cell) — read the processed parquet files and call
   `run_active_learning(...)`, passing an **XGBoost model**, the (empty)
   labeled-train data, the unlabeled pool, the test set, the tunable `params`,
   the `label_mapping`, and the `sentence transformer`.

Cold start by design: the labeled-train set is **empty** — only the test set is
labeled. The model scores zero-shot until you label enough rows. Class
imbalance is handled with **balanced class weights** inside the model.

The only project import is **`active_learner.py`** — one self-contained file.
Run `python generate_data.py` once first to create the CSVs.

In [ ]:
# ============================== CONFIG ==============================
# --- raw inputs ---
UNLABELED_TRAIN_PATH = "unlabeled_train.csv"   # pool of texts to label
LABELED_TEST_PATH    = "labeled_demo.csv"      # held-out *labeled* test set
LABELED_SAVE_PATH    = "labeled_train_al.csv"  # labels you add are written here

# --- processed feature datasets (written by the preprocess cell) ---
TRAIN_FEATURES_PATH = "train_features_dataset_processed.parquet"
TEST_FEATURES_PATH  = "test_features_dataset_processed.parquet"

TEXT_COLUMN, LABELS_COLUMN = "text", "labels"

# --- embedding model ---
EMBED_MODEL_NAME = "all-MiniLM-L6-v2"
EMBED_BATCH_SIZE = 64

# --- tunable parameters (model hyperparameters + active-learning loop) ---
params = {
    "n_estimators": 200,
    "max_depth": 6,
    "learning_rate": 0.1,
    "n_negatives_per_text": 3,
    "use_pca": False,
    "pca_components": 64,
    "feature_mode": "concat+diff+prod",
    # active-learning loop
    "retrain_every": 8,           # auto-retrain after N new labels (0 = manual)
    "query_strategy": "margin",   # "margin" | "least_confidence" | "random"
    "threshold": 0.5,
}

# --- labels (this is the label_mapping passed to the module) ---
label_dict = {
    "sentiment": {
        "positive": "expresses approval, happiness, or satisfaction",
        "negative": "expresses disapproval, anger, or frustration",
        "neutral":  "factual or descriptive with no clear emotional valence",
    },
    "topic": {
        "pricing":  "mentions cost, price, fees, value for money",
        "support":  "mentions customer service, help, agents, response time",
        "quality":  "mentions product quality, defects, durability, materials",
        "shipping": "mentions delivery, shipping speed, packaging, arrival",
    },
}
# ====================================================================

In [ ]:
# ── Preprocess: embed the text into feature columns, keep it a DataFrame ──
import json

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer

tqdm.pandas()


def coerce_labels(x):
    if isinstance(x, list):
        return [str(v) for v in x]
    if isinstance(x, str) and x.strip():
        s = x.strip()
        if s.startswith("["):
            try:
                return [str(v) for v in json.loads(s)]
            except Exception:
                pass
        return [s]
    return []


def load_csv(path):
    df = pd.read_csv(path)
    if LABELS_COLUMN in df.columns:
        df[LABELS_COLUMN] = df[LABELS_COLUMN].progress_apply(coerce_labels)
    else:
        df[LABELS_COLUMN] = [[] for _ in range(len(df))]
    return df


embedding_model = SentenceTransformer(EMBED_MODEL_NAME)


def add_embedding_features(df):
    """Return a DataFrame = [text, labels, emb_0 ... emb_d-1] (labels JSON-encoded)."""
    texts = df[TEXT_COLUMN].astype(str).tolist()
    chunks = []
    for i in tqdm(range(0, len(texts), EMBED_BATCH_SIZE), desc="embedding"):
        batch = texts[i:i + EMBED_BATCH_SIZE]
        chunks.append(np.asarray(embedding_model.encode(batch, normalize_embeddings=True)))
    embs = np.vstack(chunks)
    emb_df = pd.DataFrame(embs, columns=[f"emb_{j}" for j in range(embs.shape[1])])

    out = df.reset_index(drop=True).copy()
    out[LABELS_COLUMN] = out[LABELS_COLUMN].apply(json.dumps)   # parquet-friendly
    return pd.concat([out[[TEXT_COLUMN, LABELS_COLUMN]], emb_df], axis=1)


unlabeled_raw = load_csv(UNLABELED_TRAIN_PATH)   # the pool (unlabeled)
test_raw = load_csv(LABELED_TEST_PATH)           # labeled test

train_features_dataset_processed = add_embedding_features(unlabeled_raw)
test_features_dataset_processed = add_embedding_features(test_raw)

train_features_dataset_processed.to_parquet(TRAIN_FEATURES_PATH, index=False)
test_features_dataset_processed.to_parquet(TEST_FEATURES_PATH, index=False)

print("saved", TRAIN_FEATURES_PATH, train_features_dataset_processed.shape)
print("saved", TEST_FEATURES_PATH, test_features_dataset_processed.shape)
train_features_dataset_processed.head()

## Load the processed features and run the active-learning module

Read the parquet files saved above and hand everything to
`run_active_learning(...)`. The labeled-train set is **empty** (cold start);
the module clones the XGBoost model per fit, applies **balanced class weights**,
builds the embedding-pair features, and drives the labeling loop.

In [ ]:
import pandas as pd
import xgboost as xgb

from active_learner import run_active_learning   # single self-contained file

# Load the processed feature datasets from the previous cell.
train_features = pd.read_parquet(TRAIN_FEATURES_PATH)   # unlabeled pool
test_features = pd.read_parquet(TEST_FEATURES_PATH)     # labeled test

# Empty labeled-train set (same schema) -> start from 0 labeled examples.
empty_train = train_features.iloc[0:0].copy()

# A library XGBoost model. Tunable params come from `params`; class weights are
# applied inside the module.
xgb_model = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    n_estimators=params["n_estimators"],
    max_depth=params["max_depth"],
    learning_rate=params["learning_rate"],
    random_state=42,
)

widget = run_active_learning(
    model=xgb_model,
    train_data=empty_train,
    unlabeled_data=train_features,
    test_data=test_features,
    params=params,
    label_mapping=label_dict,
    embedding_model=embedding_model,
    labeled_save_path=LABELED_SAVE_PATH,
)
widget